In [ ]:
# General library imports.
import sys
from pathlib import Path
from itertools import islice
from collections import Counter
import random
from IPython.display import display
from datasets import load_dataset, Image

# Make project modules available when running from the notebook directory.
project_dir = str(Path.cwd().parent)
if project_dir not in sys.path:
    sys.path.append(project_dir)

# Reload local modules during interactive use.
%load_ext autoreload
%autoreload 2
%aimport -torch -sklearn -numpy -np -pandas -tensorflow

REPO_ID = "cvl-umass/RealBirdID"
REVISION = "abd12f9a663f47bc64b8345eeb54679d134762aa"  # Use a commit hash to reproduce a specific release.

import json
from huggingface_hub import hf_hub_download


In [ ]:
# Stream a small sample without downloading the full benchmark.
answerable = load_dataset(REPO_ID, "answerable", split="validation", revision=REVISION, streaming=True)
unanswerable = load_dataset(REPO_ID, "unanswerable", split="validation", revision=REVISION, streaming=True)


In [ ]:
# Use a small deterministic sample; no model or training data is required.
# This is a workflow check, not a reproduction of the paper's metrics.
limit = 100
answerable_rows = list(islice(answerable.cast_column("image", Image(decode=False)), limit))
unanswerable_rows = list(islice(unanswerable.cast_column("image", Image(decode=False)), limit))
assert answerable_rows and unanswerable_rows
print("Evaluation rows:", len(answerable_rows), len(unanswerable_rows))


In [ ]:
# Guess uniformly among the benchmark species; this baseline ignores the image.
taxonomy_path = hf_hub_download(REPO_ID, "taxonomy.json", repo_type="dataset", revision=REVISION)
taxonomy = json.loads(Path(taxonomy_path).read_text())
species_ids = [taxon["id"] for taxon in taxonomy if taxon["rank"] == "species"]
assert species_ids
rng = random.Random(42)
random_predictions_a = [rng.choice(species_ids) for row in answerable_rows]
random_predictions_ua = [rng.choice(species_ids) for row in unanswerable_rows]
abstain_predictions_a = [None] * len(answerable_rows)
abstain_predictions_ua = [None] * len(unanswerable_rows)


In [ ]:
# Score answerable species accuracy and unanswerable abstention separately.
def evaluate(predictions_a, predictions_ua):
    assert len(predictions_a) == len(answerable_rows)
    assert len(predictions_ua) == len(unanswerable_rows)
    accuracy = sum(pred == row["taxon_id"] for pred, row in zip(predictions_a, answerable_rows)) / len(answerable_rows)
    abstention = sum(pred is None for pred in predictions_ua) / len(unanswerable_rows)
    return {"answerable_accuracy": accuracy, "unanswerable_abstention": abstention,
            "balanced_success": (accuracy + abstention) / 2}

print("Uniform species guess:", evaluate(random_predictions_a, random_predictions_ua))
print("Always abstain:", evaluate(abstain_predictions_a, abstain_predictions_ua))
# Always abstaining gets no answerable cases right; guessing never abstains.
# Replace these prediction lists with model outputs to exercise the same scorer.
# For representative results, evaluate every row and report the release revision.
